In [1]:
from pathlib import Path
import shutil

DATA_DIR = Path("/Users/mac/MeterReadAI/data/images_after_handle")
IMG_DIR = DATA_DIR / "images"
LABEL_DIR = DATA_DIR / "labels"

IMG_DIR.mkdir(exist_ok=True)
LABEL_DIR.mkdir(exist_ok=True)

for file in DATA_DIR.iterdir():
    if file.is_file():
        if file.suffix.lower() in [".jpg", ".jpeg", ".png"]:
            shutil.move(str(file), str(IMG_DIR / file.name))
        elif file.suffix.lower() == ".json":
            shutil.move(str(file), str(LABEL_DIR / file.name))

print("Images:", len(list(IMG_DIR.glob("*.*"))))
print("Labels:", len(list(LABEL_DIR.glob("*.json"))))

Images: 242
Labels: 239


In [2]:
import hashlib
from collections import defaultdict
from pathlib import Path

def remove_duplicates(img_dir, label_dir):
    hash_map = defaultdict(list)
    for img_path in Path(img_dir).glob("*.*"):
        if img_path.suffix.lower() in [".jpg", ".jpeg", ".png"]:
            with open(img_path, "rb") as f:
                h = hashlib.md5(f.read()).hexdigest()
            hash_map[h].append(img_path)

    removed = 0
    for h, files in hash_map.items():
        if len(files) > 1:
            for dup_img in files[1:]:
                json_path = Path(label_dir) / f"{dup_img.stem}.json"
                dup_img.unlink()
                if json_path.exists():
                    json_path.unlink()
                removed += 1
                print(f"Removed duplicate: {dup_img.name} and {json_path.name}")

    if removed == 0:
        print("No duplicates found.")
    else:
        print(f"Total duplicates removed: {removed}")

remove_duplicates(IMG_DIR, LABEL_DIR)

No duplicates found.


In [3]:
img_stems = {f.stem for f in IMG_DIR.glob("*.*") if f.suffix.lower() in [".jpg", ".jpeg", ".png"]}
json_stems = {f.stem for f in LABEL_DIR.glob("*.json")}

missing_labels = img_stems - json_stems
missing_images = json_stems - img_stems

print(f"Total images: {len(img_stems)}")
print(f"Total labels: {len(json_stems)}")
print(f"Images missing label: {len(missing_labels)}")
print(f"Labels missing image: {len(missing_images)}")

Total images: 242
Total labels: 239
Images missing label: 3
Labels missing image: 0
